# The Intent Gap — Free Pilot v2 (expanded filter)

**v2 changes from v1:**
1. Expanded the repair-phrase list to include natural-language patterns (`but i wanted`, `you're not getting it`, `ugh`, `wait`, soft no's, etc.).
2. Scans **all** user turns, not just user turn 2.
3. Scales sample size from 10k to 50k by default (still ~5–8 minutes).
4. Also tracks **abandonment** signals — conversations that end immediately after a model response (proxy for silent dissatisfaction).

**v1 result was a finding in itself:** only 0.12% of multi-turn English conversations contained an explicit repair signal under the strict regex. v2 tests whether expanding the phrase list catches the actual long tail of natural-language frustration.

In [ ]:
!pip install -q datasets

In [ ]:
import re

# v2: many more natural-language repair patterns
REPAIR_PHRASES = [
    # explicit repair (v1)
    r"no,?\s+i\s+(?:meant|said|asked)",
    r"that(?:'s|\s+is)\s+not\s+what\s+i\s+(?:asked|wanted|meant|said)",
    r"that(?:'s|\s+is)\s+not\s+(?:right|correct|it)",
    r"you\s+(?:misunderstood|missed|don'?t\s+understand|aren'?t\s+understanding)",
    r"you'?re\s+not\s+(?:getting|listening|understanding)",
    r"let\s+me\s+(?:rephrase|clarify|try\s+again)",
    r"to\s+be\s+clear,?\s+i\s+(?:want|meant|need)",
    r"actually,?\s+i\s+(?:want|meant|need|asked)",
    r"what\s+i\s+actually\s+(?:need|want|meant|asked)",
    r"the\s+question\s+was",
    r"my\s+question\s+(?:was|is)",
    r"you\s+didn'?t\s+answer",
    r"read\s+(?:the\s+)?(?:prompt|question|message)\s+again",
    r"i\s+(?:think|don'?t\s+think)\s+you\s+(?:misunderstood|understood|got\s+it)",
    r"that\s+wasn'?t\s+the\s+(?:question|point)",
    r"i\s+asked\s+(?:about|for|how|why|what|where|when)",

    # explicit dissatisfaction (v1)
    r"this\s+is\s+(?:wrong|incorrect|not\s+what\s+i'?m\s+looking\s+for|not\s+helpful)",
    r"this\s+isn'?t\s+(?:right|what\s+i\s+(?:wanted|asked))",
    r"that(?:'s|\s+is)\s+(?:incorrect|wrong|not\s+accurate|inaccurate)",
    r"you\s+got\s+it\s+wrong",
    r"(?:not\s+useful|useless|unhelpful|not\s+helpful)",
    r"wrong\s+answer",

    # v2: natural-language soft repairs
    r"but\s+i\s+(?:wanted|asked|need|meant)",
    r"but\s+that(?:'s|\s+is)\s+not",
    r"i\s+meant",
    r"i\s+didn'?t\s+(?:ask|mean|want)",
    r"i\s+wasn'?t\s+asking",
    r"\bnot\s+quite\b",
    r"\bnope\b",
    r"^no[,.\s!?]*$",         # short "no" as a full message
    r"^wait[,.\s!?]",          # message starting with 'wait'
    r"^ugh",                    # message starting with 'ugh'
    r"^hmm",
    r"\bre[\s-]?read\b",
    r"try\s+again",
    r"do\s+(?:it|that)\s+(?:again|over)",
    r"that(?:'s|\s+is)\s+(?:still|now)\s+(?:wrong|not)",
    r"why\s+(?:are\s+you|did\s+you|do\s+you\s+keep)",
    r"stop\s+(?:doing|saying|trying)",
    r"that(?:'s|\s+is)\s+(?:still|just)\s+(?:not|wrong)",
    r"i\s+(?:already|just)\s+(?:said|told\s+you|asked)",
    r"can\s+you\s+(?:actually|just|please)",
    r"please\s+(?:just|actually)",
]

REPAIR_RE = re.compile('|'.join(f'({p})' for p in REPAIR_PHRASES), flags=re.IGNORECASE | re.MULTILINE)

def find_repair(text):
    if not text: return None
    m = REPAIR_RE.search(text)
    return m.group(0).lower().strip() if m else None

print(find_repair('but I wanted the second one'))  # should hit
print(find_repair('Wait, that\'s not right'))        # should hit
print(find_repair('Can you book a flight?'))         # None
print(len(REPAIR_PHRASES), 'phrases loaded.')

In [ ]:
from datasets import load_dataset

ds = load_dataset("allenai/WildChat-1M", split='train', streaming=True)
print('Stream opened.')

In [ ]:
import json
from collections import Counter

SAMPLE_SIZE = 50_000

n_total = 0
n_english = 0
n_long_enough = 0
n_repair_signal = 0
n_abandon_after_one = 0     # exactly 2 turns: user → assistant → end
phrase_counter = Counter()
candidates = []

for row in ds:
    if n_total >= SAMPLE_SIZE:
        break
    n_total += 1

    if row.get('language', '').lower() != 'english':
        continue
    n_english += 1

    turns = row.get('conversation', [])

    # Track abandonment: exactly one user + one assistant turn
    if len(turns) == 2:
        n_abandon_after_one += 1
        continue

    if len(turns) < 4:
        continue
    n_long_enough += 1

    # Scan ALL user turns for repair signal (not just turn 2)
    repair_found = None
    repair_turn_idx = None
    matched_phrase = None
    for i, turn in enumerate(turns):
        if turn.get('role') == 'user' and i > 0:    # skip first user turn (nothing to repair yet)
            r = find_repair(turn.get('content', ''))
            if r:
                repair_found = turn.get('content', '')
                repair_turn_idx = i
                matched_phrase = r
                break

    if not repair_found:
        continue
    n_repair_signal += 1
    phrase_counter[matched_phrase] += 1

    # The conversation immediately preceding the repair turn is the failed exchange
    prev_user = next((t['content'] for t in reversed(turns[:repair_turn_idx]) if t.get('role') == 'user'), '')
    prev_asst = next((t['content'] for t in reversed(turns[:repair_turn_idx]) if t.get('role') == 'assistant'), '')

    candidates.append({
        'conv_id': row.get('conversation_hash', ''),
        'repair_at_turn': repair_turn_idx,
        'matched_phrase': matched_phrase,
        'prev_user_prompt': prev_user[:1500],
        'prev_asst_response': prev_asst[:1500],
        'repair_turn': repair_found[:1500],
    })

print(f'Sample size:                {n_total:>8}')
print(f'English:                    {n_english:>8}')
print(f'Single-exchange (abandoned):{n_abandon_after_one:>8}')
print(f'≥4 turns:                   {n_long_enough:>8}')
print(f'Repair signal hit:          {n_repair_signal:>8}')
rate = n_repair_signal / max(n_long_enough,1) * 100
print(f'Repair rate:                {rate:.3f}% of ≥4-turn English')

In [ ]:
for phrase, count in phrase_counter.most_common(25):
    print(f'{count:>4}   {phrase}')

In [ ]:
# Inspect 5 examples
for i, c in enumerate(candidates[:5]):
    print(f'--- Example {i+1} (repair at turn {c["repair_at_turn"]}, phrase: "{c["matched_phrase"]}") ---')
    print(f'\nUSER:\n{c["prev_user_prompt"][:600]}')
    print(f'\nASSISTANT:\n{c["prev_asst_response"][:600]}')
    print(f'\nUSER REPAIR:\n{c["repair_turn"][:600]}')
    print('\n' + '='*70 + '\n')

In [ ]:
# Save outputs
funnel = {
    'sample_size': n_total,
    'english': n_english,
    'single_exchange_abandoned': n_abandon_after_one,
    'at_least_4_turns': n_long_enough,
    'repair_signal_hit': n_repair_signal,
    'rate_per_4turn_english': round(n_repair_signal / max(n_long_enough, 1) * 100, 4),
    'top_phrases': phrase_counter.most_common(30),
    'source': 'allenai/WildChat-1M (streamed sample, v2 filter)',
    'stage': 'A v2 (expanded regex, all-turn scan)',
}

with open('pilot_funnel_v2.json', 'w') as f:
    json.dump(funnel, f, indent=2)

with open('pilot_examples_v2.jsonl', 'w') as f:
    for c in candidates[:50]:
        f.write(json.dumps(c) + '\n')

print(f'Saved pilot_funnel_v2.json and pilot_examples_v2.jsonl')
print(f'Examples saved: {min(50, len(candidates))}')

## Compare v1 vs v2

If v2 catches dramatically more (e.g., 50–500 vs v1's 3), the v1 regex was too academic and the v2 list better reflects how users actually express frustration. That's itself a methodological finding worth reporting.

If v2 also catches very few, the rarity-of-explicit-repair finding holds and we lean harder into the abandonment-as-signal angle.